2. SCD: check 
    - what not changed, 
    - what changed, then chech
      - how fast
      - how lost

# 1. Cumulative Table

## Knowing your Consumer
- **Data analysts / Data sicentists:**
  - Should be very easy to query. Not many complex data types
  - **_Flat structure_**, easy to aggreate, add calculations
- **Other data engineers:**
  - Should be compact and probably harder to query
  - **_Nested type_**, e.g. struct, array
- **ML models:**
  - Depends on the model and how its trained
  - Need **_flat structure_** with identifier, numerical features
- **Customers:**
  - Should be very easy to interpret chart

## OLTP vs. OLAP vs. Master Data
- 1. **OLTP:** optimized for low-latency, **_low volume_** queries (more for sofware enginers not DE)
  - 3NF
  - Constraints
  - PK, FK
- 2. **Master Data:** Optimzed for completeness of entity definitions, deduped
- 3. **OLAP:** Optimized for **_large volumn_**
  - GROUP BY queries
  - Minimizes JOINs

![](./images/1/OLTP_OLAP_Continuum.png)

**Notes:**
- (1) Production has all transactional data
- (2) Merge all transactional data to create Master Data = 1 table for each entity
  - Dedup and normalize
- (3) Denormalize to build dimensional model (OLAP) for slice and dice
- (4) Aggregate OLAP data + formulas to build metrics

## Cumulative Table Design
- **Core components:**
  - 2 dataframes (yesterday and today)
  - **_FULL OUTER JOIN_** the two data frames together --> **_MERGE_**
  - **_COALESCE_** values to keep everything aroung --> Get what matched and not matched
  - Hand onto all of history

![](./images/1/Cumulative_Table_Design.png)

- **Usages:** keep all history
  - Growth analytics, e.g. users active --> not active, then we only need analyze on active user
  - State transition tracking, e.g. tracking number of active user

- **Strengths:**
  - Historical analysis **_without shuffle_** (no Groupby)
  - Easy "transition" analysis, e.g. when the status changed
- **Drawbacks:**
  - Can only be **_backfilled sequentially_**
  - Handling **_PII data can be a mess_**, e.g. deleted/inactive users get carried forward


## Compactness vs. Usability Tradoff
- **The most usable tables usually**
  - Have no complex data types
  - Have identifiers
  - Easily can be manipulated with WHERE and GROUP BY
  - --> When analytics (OLAP) is the main consumer and the majority of consumers are less technical
- **The most compact tables (not human readable)**
  - Have identifiers also
  - Are **_compressed_** to be as small as possible and **_can't be queried directly_** until they're decoded
    - Many columns have big blob of bits
    - For applications, it is good because it save costs for I/O and network to send/recieve data. Applications just need to decode the data at its end.
    - For analytics, it's bad because it has to decode every rows
  - --> Online system where latency data and volumes matter a lot. Consumers are usually highly technical
- **The middle-ground tables**
  - Use complex data types (e.g. ARRAY, MAP and STRUCT), making querying trickier but also compacting more
  - --> Upstream stagging / master data where the majority of consumers are other DE

## Struct vs. Array vs. Map
- **Struct:**
  - Keys are **_rigidly defined_**, compression is good
  - Values can be **_any type_**
  - Can be thought as **_table inside table_**
- **Map:**
  - Keys are **_loosely defined_**, compression is ok
  - Values all have to be the **_same type_**
- **Array:**
  - Ordinal
  - List of values that all have to be the **_same type_**
  - Can be nested as **_array of map_** or **_array of struct_**

## Temporal Cardinality Explosions of Dimensions
- When you add a temporal aspect to your dimensions and the cardinality increase by at least 1 order of magnitude
- **Example:** Airbnb has ~6 million listings:
  - If we want ot know the nightly pricing and available of each night for the next year
    - That's 365 * 6 million = ~ 2 billion nights
  - Should this dataset be:
    - **_Lising-level_** with an array of nights?
    - **_Listing night level_** with 2 billion rows?
  - If you do the **_sorting right_**, **_Parquet_** will keep thses two about **_same size_**

### Badness of Denormalized Temporal Dimensions
If you **_explode_** it out and need to **_join other dimensions_** --> **_Spark shuffle_** will **_ruin_** the compression!

### Run-length Encoding Compression
- Probably the mose important compression technique in big data right now
  - It's why Parquet file format has become so successful
- **_Shuffle can ruin_** this. **BE CAREFUL!**
  - Shuffle happens in distributed environments when you do **JOIN** and **GROUP BY**

**Example:**

**(1) Before:** there are duplicates data such as "**_A.C. Green_**"

![](./images/1/Encodeing_Compression_Example_Before.png)

**(2) After:** what run-length encoding does

![](./images/1/Encodeing_Compression_Example_After.png)

**Notes:**
- It recognizes the duplicates
- Store the first record as "5" for every column that has duplicates
  - Not store other duplicates
  - It means the systems knows it duplicates 5 times = 4 compressions

**(3) Spark Shuffle:** After a join, Spark may **_mix up the ordering_** of rows and **_ruin your compression_**

![](./images/1/Spark_Shuffle_Example.png)

**Notes:**
- The value "**_A.C. Green_**" is not compressed anymore
  - Only the last 2 records are compressed (which are store as "2" = duplicates 2 times = 1 compression)
- Output dataset is **_way much bigger_**

### Solutions for Spark Shuffle
**Way 1:** everybody in downstream need to **_re-sort_** data before joining --> it's very hard

**Way 2:** keep everything **_in array_**
  - In previous example, we:
    - (1) Keep **_player_name_** column
    - (2) Move **_season_** colum into **_array_**
    - (3) After join, we explode **_season_** columns

## Lab

![](./images/1/Lab1_player_seasons_Example.png)

**Notes:**
- We will try to get each row = **_1 player_** with the information of all **_seasons_**
  - Columns which are **_changed_** according to **_season_** = put into an **_array_**
  - Columns which are **_not changed_** = keep the same


### Preparation

![](./images/1/Lab1_Preparation.png)

**Notes:**
- (1) Create **_Array type_** to store columns changing according to season
- (2) Create **_Enum type_** for calculation
- (3) Create new table for each player
  - **_season_stats_** column has datatype we create in (1)
  - **_scoring_class_** column has datatype we create in (2)

In [0]:
%sql
CREATE TYPE season_stats AS (
                         season Integer,
                         pts REAL,
                         ast REAL,
                         reb REAL,
                         weight INTEGER
                       );
                      
 CREATE TYPE scoring_class AS
     ENUM ('bad', 'average', 'good', 'star');


 CREATE TABLE players (
     player_name TEXT,
     height TEXT,
     college TEXT,
     country TEXT,
     draft_year TEXT,
     draft_round TEXT,
     draft_number TEXT,
     season_stats season_stats[],
     scoring_class scoring_class,
     years_since_last_active INTEGER,
     is_active BOOLEAN,
     current_season INTEGER,
     PRIMARY KEY (player_name, current_season)
 );

### Script

![](./images/1/Lab1_Script1.png)

**Notes:**
- (1) CTE for new table and existing table
  - We need to manually input **_current_season_** and **_season_** for every year
- (2) Use **_COALESCE_** to get data for list of **_unchanged columns_**
- (3) Combine **_array_** of **_changed colums_**
- (4) Calculate how many year that a player **_had not played_**
  - For example, Michael Jordan played from 1996-1997 and 2000-2001
    
    ![](./images/1/Step_4_Example.png)
- (5) Use **_FULL JOIN_** to get all seasons

In [0]:
%sql
WITH last_season AS (
    SELECT * FROM players
    WHERE current_season = 2001
), this_season AS (
     SELECT * FROM player_seasons
    WHERE season = 2002
)

INSERT INTO players
SELECT
    COALESCE(ls.player_name, ts.player_name) as player_name,
    COALESCE(ls.height, ts.height) as height,
    COALESCE(ls.college, ts.college) as college,
    COALESCE(ls.country, ts.country) as country,
    COALESCE(ls.draft_year, ts.draft_year) as draft_year,
    COALESCE(ls.draft_round, ts.draft_round) as draft_round,
    COALESCE(ls.draft_number, ts.draft_number) as draft_number,    
    CASE
	     WHEN ls.season_stats IS NULL THEN
	            ARRAY[ROW(
	            ts.season,
	            ts.pts,
	            ts.ast,
	            ts.reb, 
	            ts.weight)::season_stats]
		WHEN ts.season IS NOT NULL THEN
			ls.season_stats || ARRAY[ROW(
					            ts.season,
					            ts.pts,
					            ts.ast,
					            ts.reb, 
					            ts.weight)::season_stats]
		ELSE ls.season_stats
	 END AS season_stats,
	 CASE
         WHEN ts.season IS NOT NULL THEN
             (CASE WHEN ts.pts > 20 THEN 'star'
                WHEN ts.pts > 15 THEN 'good'
                WHEN ts.pts > 10 THEN 'average'
                ELSE 'bad' END)::scoring_class
         ELSE ls.scoring_class
     END as scoring_class,
     CASE
	     WHEN ts.season IS NOT NULL THEN 0
	     ELSE ls.years_since_last_active + 1
	 END AS years_since_last_active,
     ts.season IS NOT NULL as is_active,
     COALESCE(ts.season, ls.current_season + 1) AS current_season

FROM last_season ls
FULL OUTER JOIN this_season ts
    ON ls.player_name = ts.player_name

In [0]:
%sql
SELECT *
FROM players
WHERE player_name = 'Michael Jordan'

# 2. Slowly Changed Dimension

## Idempotent Pipeline Are CRITICAL
Your pipeline produces the **_same results_** 
- Regardless of **_when_** it runs
- Regardless of **_how many times_** it runs

If not idempotent, we might:
- Get **_non-reporducible_** data
- Produce **_inconsistent_** data even there is **_no failure_**

### Causes of Pipeline not Idempotent
- **_INSERT INTO_** without **_TRUNCATE_** = generates duplicates.
  - --> **_MERGE_** = avoids duplicates
  - --> **_INSERT OVERWRITE_** = overwrite existing data with new
- Using **_START_DATE >_** without **_ENDATE <_**
  - --> Use window period to control the days of data added
- Not using a **_full set_** of partition sensors
  - Pipepline might run when there is no/partial data
- Not using **_depends_on_past_** for cumulative pipelines
  - --> Pipeline must run **_in sequence_**, not parallel
- Relying on the **_"latest" partition_** of a **_not properly modeled SCD table_**
  - Cumulative table design **_AMPLIFIES_** this bug
  - --> Track all data inflows/outflows to get correct partitions

### Consequences of Pipeline not Idempotent
- Backfilling causes **_inconsistenceis_** between the old and restated data
  - = Data always be new
- Very hard to troubleshoot bugs
- Unit testing **_cannot replicate_** the production behaviour
- Silent failure

## How Model Dimensions That Change
- Singular snapshots:
  - Be careful since these are **_not idempotent_**
- Daily partitioned snapshots
- SCD Types 0,1,2

### Which Type Are Idempotent
- **Type 0** and **Type 2** **_are idempotent_**:
  - Type 0 is because the values are **_unchanging_**
  - Type 2 is because you need to be careful to use start_date, end_date in syntax
- **Type 1** is **_not_** idempotent:
  - If you backfill with this dataset, you will get the dimension as it is not, not as it was
- **Type 3** is **_not_** idempotent
  - If you backfill with this dataset, it's **_impossible_** to know when to pick "original" vs. "current"

### SCD2 Loading
- Load the **_entire history_** in one query
  - Inefficient but numble (VN nhanh nhen)
  - 1 query and you are done
- **_Incrementally load_** the data after the previous SCD is generated
  - Has the same "depends_on_past" constraint
  - Efficient but cumbersome (VN cong kenh)

## Lab

# 3. Graph Data Modeling

## Additive vs. Non-additive dimensions
**Question:** Can an entiry have **_2+ values at the same time_** over a given period of time?

1. **Additive** = No = only 1 value --> Most of the dimension
    - **_Can_** be used with **_SUM (combining subtotals = grand total)_**
    - For example,
      - The population **=** 20 years old + 30 years old + 40 years old...
      - Car brands **=** Honda + Toyota...
2. **Non-Additive** = Yes.
    - **_Cannot_** be used with **_SUM_**
    - For example,
      - Total cars driver **!=** Honda drivers + Toyota drivers... because 1 driver can own more than 1 cars
      - Total active users **!=** User on web + Android users + iPhone user 

## The Power of Enums

**Rule of thumb:** should contain **_<= 50 values_**

**Benefits:**
- Built-in **_data quality check_**
  - Pipeline would **fail** if it does **_not fit_** datatype
- Built-in **_static fields_**
  - e.g. categorize line item data as X, Y or Z
- Built-in **_documentation_**
  - Know all **_possible values_** of a list (or a field)
- More efficient data **_sub-partitions_**
  - e.g. partition on date + enum value instead of date alone
- Enable **_thrift_** (enumerates and enforces data schemas that can be shared/understood across all systems in data pipeline - logging, ETL processes, etc.)

**Example:**
- Airbnb:
  - Unit Economics (fees, coupons, credits, insurance, infrastructure cost, taxes, etc.)
- Netflix:
  - Infrastructure Graph (applications, databases, servers, code bases, CI/CD jobs, etc.)
- Facebook:
  - Family of Apps (oculus, instagram, facebook, messenger, whatsapp, threads, etc.)

**Modeling with Enum:**

![](./images/1/Power_of_Enum_Example.png)

## Flexible Data Types = Map
- **Benefits:**
  - Don't have to run ALTER TABLE
  - Can manage a lot more columns
  - Schemas don't have ton of "NULL" columns
  - "Other_properties" column is pretty awrsome for realy-used-but-needed columns

- **Drawbacks:**
  - Compression of usually worse (especially JSON)
  - Readability, queryability

## Graph Data Modeling
Graph modeling is RELATIONSHIP focused, not ENTITY focused

![](./images/1/Graph_Data_Model.png)

![](./images/1/Vertext_Edges.png)

- **VERTEX:** **_Poor_** job at modeling the **_entities_**. Usually the model looks like:
  - Identifier: STRING
  - Type: STRING
  - Properties: MAP<STRING, STRING>
- **EDGES:** The **_relationship_** are modeled more in depth. Usually the model looks like:
  - subject_identifier: STRING
    - e.g. player name
  - subject_type: VERTEXT_TYPE
    - e.g. player
  - object_identifier: STRING
    - e.g. team name
  - object_type: VERTEXT_TYPE 
    - e.g. team
  - edge_type: EDGE_TYPE --> usually is a **_verb_**
    - e.g. play on
  - properties: MAP<STRINBG, STRING>
    - e.g. some attributes like year that player start to play for that team

![](./images/1/Graph_Diagram_Example.png)